In [ ]:
!pip uninstall -y numpy pandas pyarrow datasets


In [ ]:
!pip install numpy==1.26.4 pandas==2.1.4 pyarrow==14.0.2 datasets==2.17.0

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np

2025-12-25 09:41:06.295175: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766655666.494086     116 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766655666.553831     116 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766655667.030438     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766655667.030479     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766655667.030482     116 computation_placer.cc:177] computation placer alr

In [2]:
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Load dataset
dataset_fabsa = load_dataset("jordiclive/fabsa")
train_ds = dataset_fabsa["train"]
test_ds = dataset_fabsa["test"]

Generating train split:   0%|          | 0/7930 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1057 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [4]:
# Extract unique Aspect Labels (ignoring sentiment)
all_aspects = set()

# We iterate over the raw list to avoid tensor errors
for label_entry in train_ds['labels']:
    for aspect, sentiment in label_entry:
        all_aspects.add(aspect)

aspect_list = sorted(list(all_aspects))
num_labels = len(aspect_list)
label2id = {l: i for i, l in enumerate(aspect_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"Found {num_labels} unique categories: {aspect_list}")

Found 12 unique categories: ['Account management: Account access', 'Company brand: Competitor', 'Company brand: General satisfaction', 'Company brand: Reviews', 'Logistics rides: Speed', 'Online experience: App website', 'Purchase booking experience: Ease of use', 'Staff support: Attitude of staff', 'Staff support: Email', 'Staff support: Phone', 'Value: Discounts promotions', 'Value: Price value for money']


In [5]:
# Data Processing (Multi-Hot Encoding)
def encode_data(example):
    # Create a vector of zeros [0, 0, ... 0]
    vec = [0.0] * num_labels 
    
    # Loop through labels, get aspect, ignore sentiment
    for aspect, sentiment in example["labels"]:
        if aspect in label2id:
            idx = label2id[aspect]
            vec[idx] = 1.0
            
    # Tokenize text
    enc = tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)
    
    # Add labels to the encoding
    enc["labels"] = vec
    return enc

In [6]:
# Initialize Tokenizer
model_name = "roberta-base" # You can swap this for 'roberta-base' or 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:
# Apply processing
train_ds = train_ds.map(encode_data, batched=False)
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Apply processing
test_ds = test_ds.map(encode_data, batched=False)
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/7930 [00:00<?, ? examples/s]

Map:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [8]:
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False)

In [9]:
# Load model
bert = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# Apply LoRA
lora_cfg = LoraConfig(
    r=16,          # Rank (Paper uses full fine-tuning, but r=16 is good for LoRA)
    lora_alpha=32,
    target_modules=["query", "key", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

In [11]:
model = get_peft_model(bert, lora_cfg)
model.to(device)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): RobertaForSequenceClassification(
      (roberta): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(50265, 768, padding_idx=1)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): Mod

In [12]:
# Optimizer
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [13]:
loss_fn = nn.BCEWithLogitsLoss()

In [14]:
epochs = 10 # FABSA paper suggests training until convergence (usually 5-10 epochs)

print("\nStarting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].float().to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")


Starting Training...


Epoch 1: 100%|██████████| 496/496 [02:00<00:00,  4.10it/s]


Epoch 1 Loss: 0.2595


Epoch 2: 100%|██████████| 496/496 [02:12<00:00,  3.75it/s]


Epoch 2 Loss: 0.1722


...

Epoch 8 Loss: 0.1192


Epoch 9: 100%|██████████| 496/496 [02:12<00:00,  3.74it/s]


Epoch 9 Loss: 0.1152


Epoch 10: 100%|██████████| 496/496 [02:12<00:00,  3.74it/s]

Epoch 10 Loss: 0.1102


In [15]:
model.eval()
y_true = []
y_pred = []

print("\nStarting Evaluation...")
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Sigmoid activation -> Probability
        probs = torch.sigmoid(logits)
        
        # Threshold at 0.5 (Standard for multi-label)
        preds = (probs > 0.3).int().cpu().numpy()
        labels = batch["labels"].cpu().numpy()
        
        y_true.extend(labels)
        y_pred.extend(preds)

# Calculate Metrics
weighted_f1 = f1_score(y_true, y_pred, average="weighted")
precision = precision_score(y_true, y_pred, average="micro")
recall = recall_score(y_true, y_pred, average="micro")


y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

print(f"Weighted F1:  {weighted_f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")


Starting Evaluation...


100%|██████████| 100/100 [00:12<00:00,  7.75it/s]

Weighted F1:  0.8125
Precision: 0.7789
Recall:    0.8481


In [16]:
y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)

sample_accuracies = []
for t, p in zip(y_true_arr, y_pred_arr):
    correct = (t * p).sum()               # count correctly predicted labels
    total = t.sum()                       # total actual labels for that sample
    if total == 0:                         # if no true labels exist
        sample_accuracies.append(1.0)      
    else:
        sample_accuracies.append(correct / total)

overall_label_accuracy = np.mean(sample_accuracies)
print(f"Label-wise Sample Accuracy: {overall_label_accuracy:.4f}")


Label-wise Sample Accuracy: 0.8729
